# LC 211 — Design Add and Search Words Data Structure

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> A Trie handles exact matches
in O(m). When '.' wildcards are introduced, use DFS at each
dot node — try every child branch and return True if any
branch completes the remaining pattern.
</div>

## Official Problem Statement

Design a data structure that supports adding new words and
finding if a string matches any previously added string.

Implement the `WordDictionary` class:

- `WordDictionary()` — initializes the object.
- `void addWord(word)` — adds `word` to the data structure;
  it can be matched later.
- `bool search(word)` — returns `true` if there is any
  string in the data structure that matches `word`, or
  `false` otherwise. `word` may contain dots `'.'` where
  dots can be matched with **any** letter.

**Constraints:**
- `1 <= word.length <= 25`
- `word` in `addWord` consists of lowercase English letters.
- `word` in `search` consists of `'.'` or lowercase English
  letters.
- There will be at most `2` dots in `search` words.
- At most `10^4` calls to `addWord` and `search`.

## What This Is Actually Asking

This is LC 208 (Implement Trie) with one extension: the
search query can contain wildcard `'.'` characters that
match any single letter.

For normal characters the walk is identical to a standard
Trie; for `'.'` we must branch and try every child that
exists at that level.

Because there are at most 2 dots and 26 possible branches
per dot, the worst-case branching is bounded (26^2 = 676
paths), so it doesn't explode in practice.

The natural implementation is a recursive DFS helper
that takes the current node and the remaining suffix of
the search pattern.

## Walk Through an Example by Hand

Operations: addWord("bad"), addWord("dad"), addWord("mad"),
search("pad"), search("bad"), search(".ad"),
search("b..")

**After three addWords the trie looks like:**
```
root -> 'b' -> 'a' -> 'd'[END]
     -> 'd' -> 'a' -> 'd'[END]
     -> 'm' -> 'a' -> 'd'[END]
```

**search("pad")** → root has no child 'p' → **False**

**search("bad")** → root→'b'→'a'→'d', is_end=True
→ **True**

**search(".ad")** → at root, '.' triggers DFS over all
children {b,d,m}:
  - try 'b': node→'a'→'d', is_end=True → **True** (stop)

**search("b..")** → root→'b' (normal), then '.' over
children of 'b' = {'a'}:
  - try 'a': node→ then '.' over children of 'a' = {'d'}:
    - try 'd': is_end=True → **True**

## The Picture

After: addWord("bad"), addWord("dad"), addWord("mad")

```
              root
           /   |   \
         'b'  'd'  'm'
          |    |    |
         'a'  'a'  'a'
          |    |    |
         'd'* 'd'* 'd'*

  * = is_end True
```

search(".ad") — dot at position 0:
```
  Try each child of root:
    'b' -> 'a' -> 'd'  is_end? YES -> True
    (short-circuit: no need to try 'd' or 'm')
```

search("b..") — dot at positions 1 and 2:
```
  root -> 'b' (match)
    Try each child of 'b': only 'a'
      'a' matched by '.'
        Try each child of 'a': only 'd'
          'd' matched by '.', is_end? YES -> True
```

## When To Use This Pattern

- When you need **wildcard or pattern matching** over a
  stored set of strings, think **Trie + DFS**.
- When patterns have a bounded number of wildcards and
  performance is critical, think **Trie + DFS** (bounded
  branching keeps it fast).
- When you must match shell-style globs (`*.log`) or
  regex-lite patterns over a dictionary, think
  **Trie + DFS**.
- When the non-wildcard characters allow early pruning
  (most branches die immediately), think **Trie + DFS**.
- When full regex is too heavy and a simple dot-wildcard
  is all that's needed, think **Trie + DFS**.

## The Approach

Use the same TrieNode (children dict + is_end flag) as
LC 208; `addWord` is identical to `insert`.

For `search`, write a recursive helper `_dfs(node, i)`
that processes the pattern starting at index `i`. If
`i == len(word)` return `node.is_end`. If `word[i]` is a
normal character, look it up in children and recurse (or
return False if absent). If `word[i] == '.'`, iterate over
all children and return True if any recursive call returns
True.

This cleanly separates the wildcard logic from the normal
walk and handles nested dots without extra bookkeeping.

In [ ]:
# Standard type hints
from typing import Dict

In [ ]:
def test_harness(cls):
    """
    Replay (op, args, expected) sequences.
    'None' expected values are skipped (constructor /
    addWord which returns void).
    """
    sequences = [
        # LeetCode example
        [
            ("WordDictionary", [],         None),
            ("addWord",        ["bad"],    None),
            ("addWord",        ["dad"],    None),
            ("addWord",        ["mad"],    None),
            ("search",         ["pad"],    False),
            ("search",         ["bad"],    True),
            ("search",         [".ad"],    True),
            ("search",         ["b.."],    True),
        ],
        # Dot at start and end
        [
            ("WordDictionary", [],         None),
            ("addWord",        ["at"],     None),
            ("addWord",        ["and"],    None),
            ("addWord",        ["an"],     None),
            ("addWord",        ["add"],    None),
            ("search",         [".at"],    False),
            ("search",         ["a."],     True),
            ("search",         ["a.d"],    True),
            ("search",         [".nd"],    True),
            ("search",         ["."],      False),
        ],
        # Empty-ish and exact match only
        [
            ("WordDictionary", [],         None),
            ("addWord",        ["a"],      None),
            ("search",         ["."],      True),
            ("search",         ["a"],      True),
            ("search",         ["b"],      False),
        ],
    ]

    passed = 0
    failed = 0

    for s_idx, seq in enumerate(sequences):
        obj = None
        print(f"--- Sequence {s_idx + 1} ---")
        for op, args, expected in seq:
            if op == "WordDictionary":
                obj = cls()
                result = None
            else:
                result = getattr(obj, op)(*args)

            if expected is None:
                print(f"  {op}({args}) -> (void)")
                continue

            ok = result == expected
            status = "PASSED" if ok else "FAILED"
            if ok:
                passed += 1
            else:
                failed += 1
            print(
                f"  {status} | {op}({args})"
                f" expected={expected} got={result}"
            )

    total = passed + failed
    print(f"\nResult: {passed}/{total} passed, "
          f"{failed}/{total} failed.")

In [ ]:
class TrieNode:
    """
    A single node in the Trie.

    Attributes
    ----------
    children : dict[str, TrieNode]
        Maps a character to its child TrieNode.
    is_end : bool
        True if this node marks the end of a complete
        inserted word.
    """
    def __init__(self):
        self.children: Dict[str, "TrieNode"] = {}
        self.is_end: bool = False


class WordDictionary:
    """
    Trie-based dictionary supporting wildcard search.

    '.' in a search pattern matches any single letter.
    Normal characters must match exactly.

    Methods
    -------
    addWord(word: str) -> None
        Insert word into the dictionary.
    search(word: str) -> bool
        Return True if word (possibly with '.'
        wildcards) matches any stored word.
    """

    def __init__(self):
        """
        Initialize with an empty root TrieNode.
        """
        print("[DEBUG] WordDictionary initialized.")
        self.root = TrieNode()

    def addWord(self, word: str) -> None:
        """
        Insert word into the trie.

        Walk from root, creating child nodes for each
        character as needed, then mark is_end=True.

        Parameters
        ----------
        word : str
            Lowercase letters only; no wildcards.
        """
        print(f"[DEBUG] addWord('{word}')")
        pass

    def search(self, word: str) -> bool:
        """
        Return True if word matches any stored word.

        Uses a recursive DFS helper. For each position:
        - Normal char: follow child if it exists.
        - '.': try every child recursively; return True
          if any branch succeeds.
        Base case: index == len(word), return is_end.

        Parameters
        ----------
        word : str
            May contain '.' wildcard characters.

        Returns
        -------
        bool
        """
        print(f"[DEBUG] search('{word}')")
        pass

    def _dfs(self, node: TrieNode, i: int,
             word: str) -> bool:
        """
        Recursive DFS matching word[i:] from node.

        Parameters
        ----------
        node : TrieNode
            Current trie node.
        i : int
            Current index in word.
        word : str
            Full search pattern.

        Returns
        -------
        bool
        """
        print(
            f"[DEBUG] _dfs(i={i}, "
            f"remaining='{word[i:]}')")
        pass

In [ ]:
# Uncomment and run when solution is ready
# test_harness(WordDictionary)

## Complexity

Let **m** = word length, **n** = words stored,
**d** = number of dots in search pattern.

| Approach | addWord | search (no dot) | search (d dots) | Space |
|----------|---------|-----------------|-----------------|-------|
| Linear scan list | O(m) | O(n·m) | O(n·m) | O(n·m) |
| **Trie + DFS (optimal)** | **O(m)** | **O(m)** | **O(26^d · m)** | **O(n·m)** |

- With at most 2 dots, worst case is 26^2 = 676 DFS paths,
  each of length at most 25 → ~16 900 node visits max.
- Non-dot characters prune branches immediately, so the
  average case is much faster.
- Space is the same as a standard Trie: O(n·m) nodes.

## Real World Connection

**Citi** compliance teams run pattern-based searches over
transaction identifiers; a dot-wildcard Trie lets analysts
query "find all accounts matching `ACC.123.`" without
full-table regex scans, cutting query time from seconds to
milliseconds.

**AWS CloudWatch** log insights and Athena support wildcard
column-name filters; internally, matching patterns like
`event_.type` against a schema catalog uses a structure
equivalent to a wildcard Trie search.

**Data engineering** pipelines often need to classify
incoming data fields by pattern (e.g., `customer_.id`
matches `customer_v1_id` and `customer_v2_id`); a Trie
with wildcard DFS handles this in a single pass over the
pattern library.

Network intrusion detection systems match packet payload
signatures using patterns with wildcards — the same
bounded-wildcard Trie approach keeps latency predictable
even under high packet rates.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra